# Baseline DiD notebook

This notebook implements a **first-pass baseline DiD** aligned with the proposal's next step:

1. Rent growth
2. Rent growth + controls
3. Rent growth + time FE + municipality FE
4. Tenancy agreements per 10k
5. Tenancy agreements per 10k + controls
6. Tenancy agreements per 10k + time FE + municipality FE

The treatment variable is the staggered quarterly `treated` indicator. Standard errors are clustered at the municipality level, consistent with the proposal.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "src").exists():
            return candidate
    return start

ROOT = find_project_root()
OUT_DIR = ROOT / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

candidate_paths = [
    OUT_DIR / "analysis_ready_panel_fixed.pkl",
    OUT_DIR / "analysis_ready_panel_fixed.jsonl",
    OUT_DIR / "analysis_ready_panel.pkl",
    OUT_DIR / "analysis_ready_panel.jsonl",
]

panel_path = next((p for p in candidate_paths if p.exists()), None)
if panel_path is None:
    raise FileNotFoundError(
        "Could not find an analysis-ready panel in ROOT/out. "
        f"Tried: {[p.name for p in candidate_paths]}"
    )

if panel_path.suffix == ".pkl":
    panel = pd.read_pickle(panel_path)
else:
    panel = pd.read_json(panel_path, lines=True)

print("ROOT:", ROOT)
print("Loaded:", panel_path)
print("Shape:", panel.shape)
panel.head()

ROOT: /home/venatio/documents/university/public-sector-2026
Loaded: /home/venatio/documents/university/public-sector-2026/out/analysis_ready_panel.pkl
Shape: (56908, 61)


,Code,Municipality,municipality_norm,Year,Quarter,quarter_num,time_index,Prices,Contracts,price_observed_raw,contracts_observed_raw,price_imputed,contracts_imputed,has_any_observed_price,has_any_observed_contracts,exclude_from_price_analysis,exclude_from_contract_analysis,has_valid_price_for_analysis,has_valid_contracts_for_analysis,Treated 1,Treated 2,Control,ever_treated,cohort,first_treat_period,first_treat_index,treated,event_time,pop_total,pop_males,pop_females,pop_native_total,pop_native_males,pop_native_females,pop_foreign_total,pop_foreign_males,pop_foreign_females,mig_total,mig_total_males,mig_total_females,mig_inflows,mig_inflows_males,mig_inflows_females,mig_outflows,mig_outflows_males,mig_outflows_females,income_tax_base,income_tax_liability,log_prices,price_growth_qoq_pct,price_growth_yoy_pct,log_price_growth_qoq,log_price_growth_yoy,contracts_per_10k,contracts_per_10k_qoq_pct,contracts_per_10k_yoy_pct,rent_to_income_proxy,annualized_rent_to_income_proxy,rent_to_tax_liability_proxy,outcome_price_growth,outcome_market_activity
0,00005,NaN,,2015,Q1,1,8061,NaN,NaN,0,0,0,0,0,0,1,1,0,0,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00005,NaN,,2015,Q2,2,8062,NaN,NaN,0,0,0,0,0,0,1,1,0,0,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00005,NaN,,2015,Q3,3,8063,NaN,NaN,0,0,0,0,0,0,1,1,0,0,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00005,NaN,,2015,Q4,4,8064,NaN,NaN,0,0,0,0,0,0,1,1,0,0,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,02050,NaN,,2011,Q1,1,8045,NaN,NaN,0,0,0,0,0,0,1,1,0,0,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Analysis window

The proposal notes that a shorter and cleaner pre-period such as 2022–2023 may be preferable in robustness checks.

In [3]:
ANALYSIS_START_YEAR = 2022
ANALYSIS_END_YEAR = None
PRE_REFERENCE_INDEX = 2024 * 4 + 2 # 2024-Q2, used for pre-policy baseline controls

In [4]:
panel = panel.copy()
panel.columns = [str(c).strip() for c in panel.columns]

required_cols = [
    "Code", "Municipality", "Year", "Quarter", "treated",
    "outcome_price_growth", "outcome_market_activity",
    "rent_to_income_proxy", "contracts_per_10k", "price_growth_yoy_pct"
]
missing_required = [c for c in required_cols if c not in panel.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

quarter_map = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4}
if "quarter_num" not in panel.columns:
    panel["quarter_num"] = panel["Quarter"].map(quarter_map).astype("Int64")
if "time_index" not in panel.columns:
    panel["time_index"] = pd.to_numeric(panel["Year"], errors="coerce").astype("Int64") * 4 + panel["quarter_num"]

panel["Code"] = panel["Code"].astype(str).str.zfill(5)
panel["Municipality"] = panel["Municipality"].astype(str)
panel["Year"] = pd.to_numeric(panel["Year"], errors="coerce").astype("Int64")
panel["treated"] = pd.to_numeric(panel["treated"], errors="coerce").fillna(0).astype(int)

for c in [
    "outcome_price_growth", "outcome_market_activity",
    "rent_to_income_proxy", "contracts_per_10k", "price_growth_yoy_pct",
    "has_valid_price_for_analysis", "has_valid_contracts_for_analysis"
]:
    if c in panel.columns:
        panel[c] = pd.to_numeric(panel[c], errors="coerce")

panel = panel[panel["Year"].notna()].copy()
panel = panel[panel["Year"] >= ANALYSIS_START_YEAR].copy()
if ANALYSIS_END_YEAR is not None:
    panel = panel[panel["Year"] <= ANALYSIS_END_YEAR].copy()

panel["period"] = panel["Year"].astype(int).astype(str) + "-" + panel["Quarter"].astype(str)
panel = panel.sort_values(["Code", "Year", "quarter_num"]).reset_index(drop=True)

panel[["Code", "Municipality", "Year", "Quarter", "treated", "outcome_price_growth", "outcome_market_activity"]].head()

,Code,Municipality,Year,Quarter,treated,outcome_price_growth,outcome_market_activity
0,08001,Abrera,2022,Q1,0,10.1118,58.2815
1,08001,Abrera,2022,Q2,0,5.2855,48.8304
2,08001,Abrera,2022,Q3,0,6.7388,33.8663
3,08001,Abrera,2022,Q4,0,-8.0805,40.9546
4,08001,Abrera,2023,Q1,0,-20.4975,60.2787


## Pre-policy baseline controls

To match your requested table, the control specifications use municipality-level **pre-policy means** computed before **2024-Q2**:

- for rent-growth models: pre-policy **tenancy agreements per 10k**
- for tenancy-agreement models: pre-policy **rent growth**
- the rent/income proxy enters contemporaneously, as requested

In [5]:
pre_policy = panel["time_index"] < PRE_REFERENCE_INDEX

baseline_controls = (
    panel.loc[pre_policy, ["Code", "contracts_per_10k", "price_growth_yoy_pct"]]
    .groupby("Code", as_index=False)
    .agg(
        pre_contracts_per_10k=("contracts_per_10k", "mean"),
        pre_rent_growth=("price_growth_yoy_pct", "mean"),
    )
)

panel = panel.merge(baseline_controls, on="Code", how="left")

panel[["Code", "Municipality", "pre_contracts_per_10k", "pre_rent_growth"]].head()

,Code,Municipality,pre_contracts_per_10k,pre_rent_growth
0,08001,Abrera,43.1480,6.0860
1,08001,Abrera,43.1480,6.0860
2,08001,Abrera,43.1480,6.0860
3,08001,Abrera,43.1480,6.0860
4,08001,Abrera,43.1480,6.0860


## Regression samples

- **Rent-growth models** use the valid price-analysis sample.
- **Tenancy-agreement models** use the valid contracts-analysis sample.

In [6]:
if "has_valid_price_for_analysis" in panel.columns:
    rent_sample = panel[panel["has_valid_price_for_analysis"] == 1].copy()
else:
    rent_sample = panel.copy()

if "has_valid_contracts_for_analysis" in panel.columns:
    contracts_sample = panel[panel["has_valid_contracts_for_analysis"] == 1].copy()
else:
    contracts_sample = panel.copy()

rent_sample = rent_sample[rent_sample["outcome_price_growth"].notna()].copy()
contracts_sample = contracts_sample[contracts_sample["outcome_market_activity"].notna()].copy()

print("Rent sample rows:", len(rent_sample), "| Municipalities:", rent_sample["Code"].nunique())
print("Contracts sample rows:", len(contracts_sample), "| Municipalities:", contracts_sample["Code"].nunique())

Rent sample rows: 8448 | Municipalities: 528
Contracts sample rows: 15152 | Municipalities: 947


## Estimate baseline DiD models

In [7]:
def fit_clustered_ols(df: pd.DataFrame, formula: str, cluster_col: str = "Code"):
    work = df.copy()
    model = smf.ols(formula=formula, data=work)
    return model.fit(
        cov_type="cluster",
        cov_kwds={"groups": work[cluster_col]},
        use_t=True,
    )

m1 = fit_clustered_ols(rent_sample, "outcome_price_growth ~ treated")
m2 = fit_clustered_ols(
    rent_sample.dropna(subset=["rent_to_income_proxy", "pre_contracts_per_10k"]),
    "outcome_price_growth ~ treated + rent_to_income_proxy + pre_contracts_per_10k"
)
m3 = fit_clustered_ols(
    rent_sample,
    "outcome_price_growth ~ treated + C(period) + C(Code)"
)

m4 = fit_clustered_ols(contracts_sample, "outcome_market_activity ~ treated")
m5 = fit_clustered_ols(
    contracts_sample.dropna(subset=["rent_to_income_proxy", "pre_rent_growth"]),
    "outcome_market_activity ~ treated + rent_to_income_proxy + pre_rent_growth"
)
m6 = fit_clustered_ols(
    contracts_sample,
    "outcome_market_activity ~ treated + C(period) + C(Code)"
)

models = {
    "1. Rent growth": m1,
    "2. Rent growth + controls": m2,
    "3. Rent growth + time FE + muni FE": m3,
    "4. Tenancy agreements": m4,
    "5. Tenancy agreements + controls": m5,
    "6. Tenancy agreements + time FE + muni FE": m6,
}

In [8]:
def stars(p):
    if pd.isna(p):
        return ""
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""

def fmt_coef(res, name):
    if name not in res.params.index:
        return ""
    return f"{res.params[name]:.4f}{stars(res.pvalues[name])}"

def fmt_se(res, name):
    if name not in res.bse.index:
        return ""
    return f"({res.bse[name]:.4f})"

def n_munis(res):
    return int(res.model.data.frame["Code"].nunique())

row_spec = [
    ("Treated", "treated"),
    ("", "treated__se"),
    ("Rent/Income proxy", "rent_to_income_proxy"),
    ("", "rent_to_income_proxy__se"),
    ("Tenancy agreements per 10k, pre", "pre_contracts_per_10k"),
    ("", "pre_contracts_per_10k__se"),
    ("Rent growth, pre", "pre_rent_growth"),
    ("", "pre_rent_growth__se"),
    ("Observations", "__nobs__"),
    ("Municipalities", "__nmuni__"),
    ("Time FE", "__timefe__"),
    ("Municipality FE", "__munife__"),
]

reg_table = pd.DataFrame({"Row": [x[0] for x in row_spec]})

for colname, res in models.items():
    vals = []
    for _, key in row_spec:
        if key.endswith("__se"):
            vals.append(fmt_se(res, key.replace("__se", "")))
        elif key == "__nobs__":
            vals.append(int(res.nobs))
        elif key == "__nmuni__":
            vals.append(n_munis(res))
        elif key == "__timefe__":
            vals.append("Yes" if "C(period)" in res.model.formula else "No")
        elif key == "__munife__":
            vals.append("Yes" if "C(Code)" in res.model.formula else "No")
        else:
            vals.append(fmt_coef(res, key))
    reg_table[colname] = vals

reg_table

,Row,1. Rent growth,2. Rent growth + controls,3. Rent growth + time FE + muni FE,4. Tenancy agreements,5. Tenancy agreements + controls,6. Tenancy agreements + time FE + muni FE
0,Treated,-2.9278***,-5.7301***,-4.2048***,5.5126***,-2.0853***,-6.7518***
1,,(0.4462),(0.4493),(0.7462),(0.7661),(0.7436),(0.6565)
2,Rent/Income proxy,,860.2152***,,,-298.0654***,
3,,,(55.4788),,,(101.8501),
4,"Tenancy agreements per 10k, pre",,0.0562***,,,,
5,,,(0.0144),,,,
6,"Rent growth, pre",,,,,0.6140***,
7,,,,,,(0.1264),
8,Observations,8448,8448,8448,15152,8448,15152
9,Municipalities,528,528,528,947,528,947


## Save outputs

In [9]:
OUT_CSV = OUT_DIR / "baseline_did_table.csv"
OUT_XLSX = OUT_DIR / "baseline_did_table.xlsx"
OUT_TEX = OUT_DIR / "baseline_did_table.tex"
OUT_META = OUT_DIR / "baseline_did_model_specs.json"

reg_table.to_csv(OUT_CSV, index=False)
reg_table.to_excel(OUT_XLSX, index=False)
OUT_TEX.write_text(reg_table.to_latex(index=False, escape=False), encoding="utf-8")

specs = {
    "analysis_start_year": ANALYSIS_START_YEAR,
    "analysis_end_year": ANALYSIS_END_YEAR,
    "pre_reference_index": PRE_REFERENCE_INDEX,
    "models": {name: res.model.formula for name, res in models.items()},
}
OUT_META.write_text(json.dumps(specs, indent=2), encoding="utf-8")

print("Saved:")
print(" -", OUT_CSV)
print(" -", OUT_XLSX)
print(" -", OUT_TEX)
print(" -", OUT_META)

Saved:
 - /home/venatio/documents/university/public-sector-2026/out/baseline_did_table.csv
 - /home/venatio/documents/university/public-sector-2026/out/baseline_did_table.xlsx
 - /home/venatio/documents/university/public-sector-2026/out/baseline_did_table.tex
 - /home/venatio/documents/university/public-sector-2026/out/baseline_did_model_specs.json


In [10]:
for name, res in models.items():
    print("\n" + "=" * 100)
    print(name)
    print(res.summary().tables[1])


1. Rent growth
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      5.2078      0.206     25.266      0.000       4.803       5.613
treated       -2.9278      0.446     -6.562      0.000      -3.804      -2.051

2. Rent growth + controls
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept               -16.7494      1.384    -12.101      0.000     -19.469     -14.030
treated                  -5.7301      0.449    -12.754      0.000      -6.613      -4.847
rent_to_income_proxy    860.2152     55.479     15.505      0.000     751.228     969.202
pre_contracts_per_10k     0.0562      0.014      3.907      0.000       0.028       0.084

3. Rent growth + time FE + muni FE


/home/venatio/documents/university/public-sector-2026/.venv/lib/python3.14/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 543, but rank is 16
  warnings.warn('covariance of constraints does not have full '


                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept                5.4894      0.614      8.943      0.000       4.284       6.695
C(period)[T.2022-Q2]     1.2487      0.670      1.864      0.063      -0.067       2.564
C(period)[T.2022-Q3]     2.6688      0.825      3.237      0.001       1.049       4.289
C(period)[T.2022-Q4]     4.5586      0.896      5.086      0.000       2.798       6.319
C(period)[T.2023-Q1]     3.6900      1.049      3.518      0.000       1.630       5.750
C(period)[T.2023-Q2]     3.2751      0.912      3.592      0.000       1.484       5.066
C(period)[T.2023-Q3]     1.1419      0.873      1.308      0.191      -0.573       2.857
C(period)[T.2023-Q4]     0.4098      0.856      0.479      0.632      -1.272       2.092
C(period)[T.2024-Q1]     1.2369      0.957      1.293      0.197      -0.643       3.117
C(period)[T.2024-Q2] 

/home/venatio/documents/university/public-sector-2026/.venv/lib/python3.14/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 962, but rank is 15
  warnings.warn('covariance of constraints does not have full '
